# Splunk vs ELK vs Datadog — Cost, Scale, Use Case

## Mental Model

This notebook compares three observability platform patterns through a Senior Data Engineering lens:

- **Splunk** = premium search and operations platform optimized for regulated enterprises
- **ELK** = flexible, infrastructure-funded open ecosystem optimized for customization and cost control
- **Datadog** = cloud-native SaaS optimized for fast time-to-value across metrics, traces, logs, and APM

### Dataset context used throughout

- **PostgreSQL**: `localhost:5432`, database `de_telemetry`, user `de_admin`
- **Tables**
  - `endpoints`: 10,000 rows
  - `metrics`: 500,000 rows
  - `alerts`: 25,000 rows
- **Narrative**
  - Citi-style environment with **6,000+ API endpoints** monitored for latency, error rate, and throughput
  - Alerts escalate through severity tiers

### What this notebook does

1. Reads telemetry context from PostgreSQL
2. Builds **100 observability events**
3. Attempts a **Splunk live demo** if a local Splunk endpoint is reachable
4. Runs 3 search/query scenarios and measures response time
5. Compares **Splunk vs ELK vs Datadog**
6. Applies a decision framework for a **10TB/day, regulated FinTech** scenario


In [ ]:

import os
import json
import time
import socket
import math
from datetime import datetime, timezone
from typing import Dict, Any, List, Tuple

import pandas as pd
import requests
from sqlalchemy import create_engine, text
from requests.auth import HTTPBasicAuth
from IPython.display import display, Markdown

# Core environment from prompt
PG_HOST = "localhost"
PG_PORT = 5432
PG_DB = "de_telemetry"
PG_USER = "de_admin"
PG_PASSWORD = "DeAdmin2026!"

KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_CONTAINER = "citi_kafka"
SPARK_MASTER = "local[*]"
JAVA_HOME = r"C:/Program Files/Java/jre1.8.0_481"
HADOOP_HOME = r"C:/hadoop"
AIRFLOW_URL = "http://localhost:8082"
MLFLOW_URL = "http://localhost:5000"
DBT_EXE = r"C:/py_venv/proj_educate/Scripts/dbt.exe"
DBT_PROJECT = "citi_dbt"
DATABRICKS_HOST = "https://dbc-9f35a83d-b4e7.cloud.databricks.com"
GCP_PROJECT = "citi-de-learning"
AZ_SUBSCRIPTION = "b3811436-61fc-4a3a-a6a9-deb05955076d"
AWS_PROFILE = "study"
AWS_REGION = "us-east-1"
AWS_ACCOUNT = "357811130281"

# Splunk configuration: real values are intentionally taken from environment or defaults.
# This keeps the notebook executable in multiple environments without code edits.
SPLUNK_HOST = os.getenv("SPLUNK_HOST", "localhost")
SPLUNK_HEC_PORT = int(os.getenv("SPLUNK_HEC_PORT", "8088"))
SPLUNK_MGMT_PORT = int(os.getenv("SPLUNK_MGMT_PORT", "8089"))
SPLUNK_SCHEME = os.getenv("SPLUNK_SCHEME", "http")
SPLUNK_HEC_TOKEN = os.getenv("SPLUNK_HEC_TOKEN", "changeme-hec-token")
SPLUNK_USERNAME = os.getenv("SPLUNK_USERNAME", "admin")
SPLUNK_PASSWORD = os.getenv("SPLUNK_PASSWORD", "changeme-admin-password")
SPLUNK_INDEX = os.getenv("SPLUNK_INDEX", "main")
SPLUNK_SOURCE = "notebook.telemetry"
SPLUNK_SOURCETYPE = "_json"

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}",
    pool_pre_ping=True,
)

def is_port_open(host: str, port: int, timeout: float = 0.5) -> bool:
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

def show_section(title: str):
    display(Markdown(f"## {title}"))

print("Notebook configuration loaded.")
print(f"PostgreSQL target: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"Splunk target: {SPLUNK_SCHEME}://{SPLUNK_HOST} HEC={SPLUNK_HEC_PORT} MGMT={SPLUNK_MGMT_PORT}")


In [ ]:

show_section("PostgreSQL data pull")

sql = '''
WITH latest_metrics AS (
    SELECT
        m.endpoint_id,
        MAX(CASE WHEN m.metric_name = 'latency_ms' THEN m.value END) AS latency_ms,
        MAX(CASE WHEN m.metric_name = 'error_rate' THEN m.value END) AS error_rate,
        MAX(CASE WHEN m.metric_name = 'throughput_rps' THEN m.value END) AS throughput_rps,
        MAX(m.timestamp) AS last_metric_ts
    FROM metrics m
    WHERE m.metric_name IN ('latency_ms', 'error_rate', 'throughput_rps')
    GROUP BY m.endpoint_id
),
latest_alerts AS (
    SELECT
        a.endpoint_id,
        COUNT(*) AS alert_count,
        MAX(a.created_at) AS last_alert_ts,
        MAX(a.severity) AS latest_severity
    FROM alerts a
    GROUP BY a.endpoint_id
)
SELECT
    e.endpoint_id,
    e.name,
    e.region,
    e.status,
    e.category,
    COALESCE(lm.latency_ms, 0) AS latency_ms,
    COALESCE(lm.error_rate, 0) AS error_rate,
    COALESCE(lm.throughput_rps, 0) AS throughput_rps,
    lm.last_metric_ts,
    COALESCE(la.alert_count, 0) AS alert_count,
    la.last_alert_ts,
    COALESCE(la.latest_severity, 'none') AS latest_severity
FROM endpoints e
LEFT JOIN latest_metrics lm ON e.endpoint_id = lm.endpoint_id
LEFT JOIN latest_alerts la ON e.endpoint_id = la.endpoint_id
ORDER BY COALESCE(lm.latency_ms, 0) DESC, e.endpoint_id
LIMIT 100
'''

telemetry_df = pd.read_sql(text(sql), engine)
telemetry_df["last_metric_ts"] = pd.to_datetime(telemetry_df["last_metric_ts"], errors="coerce", utc=True)
telemetry_df["last_alert_ts"] = pd.to_datetime(telemetry_df["last_alert_ts"], errors="coerce", utc=True)

print(f"Rows loaded: {len(telemetry_df):,}")
display(telemetry_df.head(10))


In [ ]:

show_section("Build 100 observability events")

def severity_from_row(row: pd.Series) -> str:
    if row["error_rate"] >= 0.10 or row["latency_ms"] >= 1200:
        return "critical"
    if row["error_rate"] >= 0.05 or row["latency_ms"] >= 700:
        return "high"
    if row["error_rate"] >= 0.02 or row["latency_ms"] >= 400:
        return "medium"
    return "low"

events: List[Dict[str, Any]] = []
ingest_ts = datetime.now(timezone.utc)

for _, row in telemetry_df.iterrows():
    severity = severity_from_row(row)
    event = {
        "event_type": "api_telemetry",
        "platform": "citi-demo",
        "business_context": "6,000+ API endpoints monitored for latency, error rate, throughput",
        "endpoint_id": int(row["endpoint_id"]),
        "endpoint_name": row["name"],
        "region": row["region"],
        "status": row["status"],
        "category": row["category"],
        "latency_ms": float(row["latency_ms"]),
        "error_rate": float(row["error_rate"]),
        "throughput_rps": float(row["throughput_rps"]),
        "alert_count": int(row["alert_count"]),
        "latest_severity": row["latest_severity"],
        "derived_severity": severity,
        "metric_timestamp": row["last_metric_ts"].isoformat() if pd.notna(row["last_metric_ts"]) else None,
        "alert_timestamp": row["last_alert_ts"].isoformat() if pd.notna(row["last_alert_ts"]) else None,
        "ingested_at": ingest_ts.isoformat(),
    }
    events.append(event)

events_df = pd.DataFrame(events)
print(f"Events prepared: {len(events_df)}")
display(events_df.head(10))


In [ ]:

show_section("Splunk connectivity preflight")

splunk_hec_up = is_port_open(SPLUNK_HOST, SPLUNK_HEC_PORT)
splunk_mgmt_up = is_port_open(SPLUNK_HOST, SPLUNK_MGMT_PORT)

preflight = pd.DataFrame([
    {"check": "splunk_hec_port", "target": f"{SPLUNK_HOST}:{SPLUNK_HEC_PORT}", "ok": splunk_hec_up},
    {"check": "splunk_mgmt_port", "target": f"{SPLUNK_HOST}:{SPLUNK_MGMT_PORT}", "ok": splunk_mgmt_up},
])

display(preflight)


In [ ]:

show_section("Splunk live — send 100 events")

def post_events_to_splunk(events: List[Dict[str, Any]]) -> pd.DataFrame:
    url = f"{SPLUNK_SCHEME}://{SPLUNK_HOST}:{SPLUNK_HEC_PORT}/services/collector/event"
    headers = {"Authorization": f"Splunk {SPLUNK_HEC_TOKEN}"}
    rows = []
    start = time.perf_counter()
    for i, event in enumerate(events, start=1):
        payload = {
            "time": time.time(),
            "host": "localhost",
            "source": SPLUNK_SOURCE,
            "sourcetype": SPLUNK_SOURCETYPE,
            "index": SPLUNK_INDEX,
            "event": event,
        }
        ok = False
        status_code = None
        error = None
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=10)
            status_code = resp.status_code
            ok = resp.ok
            if not resp.ok:
                error = resp.text[:500]
        except Exception as exc:
            error = str(exc)
        rows.append({"event_num": i, "ok": ok, "status_code": status_code, "error": error})
    elapsed = time.perf_counter() - start
    result = pd.DataFrame(rows)
    result.attrs["elapsed_seconds"] = elapsed
    return result

if splunk_hec_up:
    ingest_result_df = post_events_to_splunk(events)
else:
    ingest_result_df = pd.DataFrame([{
        "event_num": 0,
        "ok": False,
        "status_code": None,
        "error": "Splunk HEC port is not reachable; notebook continues in comparison-safe mode."
    }])
    ingest_result_df.attrs["elapsed_seconds"] = 0.0

display(ingest_result_df.head(10))
print(f"Total ingest elapsed seconds: {ingest_result_df.attrs.get('elapsed_seconds', 0.0):.4f}")
print(f"Successful posts: {int(ingest_result_df['ok'].fillna(False).sum())} / {max(len(events), 1)}")


In [ ]:

show_section("Run 3 SPL searches and measure query time")

def run_splunk_search(search_query: str, earliest: str = "-24h", latest: str = "now", max_count: int = 100) -> Tuple[Dict[str, Any], pd.DataFrame]:
    job_url = f"{SPLUNK_SCHEME}://{SPLUNK_HOST}:{SPLUNK_MGMT_PORT}/services/search/jobs"
    results_url_template = f"{SPLUNK_SCHEME}://{SPLUNK_HOST}:{SPLUNK_MGMT_PORT}/services/search/jobs/{{sid}}/results"
    auth = HTTPBasicAuth(SPLUNK_USERNAME, SPLUNK_PASSWORD)
    headers = {"Accept": "application/json"}
    started = time.perf_counter()

    create_resp = requests.post(
        job_url,
        auth=auth,
        headers=headers,
        data={
            "search": search_query,
            "earliest_time": earliest,
            "latest_time": latest,
            "output_mode": "json",
        },
        timeout=20,
    )
    create_resp.raise_for_status()
    sid = create_resp.json()["sid"]

    dispatch_state = None
    for _ in range(30):
        status_resp = requests.get(
            f"{job_url}/{sid}",
            auth=auth,
            headers=headers,
            params={"output_mode": "json"},
            timeout=20,
        )
        status_resp.raise_for_status()
        entry = status_resp.json()["entry"][0]
        content = entry["content"]
        dispatch_state = content.get("dispatchState")
        if dispatch_state == "DONE":
            break
        time.sleep(1)

    results_resp = requests.get(
        results_url_template.format(sid=sid),
        auth=auth,
        headers=headers,
        params={"output_mode": "json", "count": max_count},
        timeout=20,
    )
    results_resp.raise_for_status()
    elapsed = time.perf_counter() - started
    results_json = results_resp.json()
    results = results_json.get("results", [])
    return {
        "sid": sid,
        "dispatch_state": dispatch_state,
        "row_count": len(results),
        "elapsed_seconds": elapsed,
        "query": search_query,
    }, pd.DataFrame(results)

spl_queries = [
    'search index=main source="notebook.telemetry" event.event_type="api_telemetry" | stats count avg(event.latency_ms) as avg_latency max(event.latency_ms) as max_latency by event.region | sort -avg_latency',
    'search index=main source="notebook.telemetry" event.event_type="api_telemetry" event.derived_severity IN ("critical","high") | stats count by event.derived_severity event.category | sort -count',
    'search index=main source="notebook.telemetry" event.event_type="api_telemetry" | top limit=10 event.endpoint_name'
]

search_summaries = []
search_results: List[Tuple[str, pd.DataFrame]] = []

if splunk_mgmt_up:
    for i, q in enumerate(spl_queries, start=1):
        try:
            meta, df = run_splunk_search(q)
            meta["query_num"] = i
            search_summaries.append(meta)
            search_results.append((f"Query {i}", df))
        except Exception as exc:
            search_summaries.append({
                "query_num": i,
                "sid": None,
                "dispatch_state": "ERROR",
                "row_count": 0,
                "elapsed_seconds": None,
                "query": q,
                "error": str(exc),
            })
            search_results.append((f"Query {i}", pd.DataFrame()))
else:
    # Safe local fallback so the notebook still executes end-to-end without raising.
    fallback_1 = (
        events_df.groupby("region", dropna=False)
        .agg(count=("endpoint_id", "count"), avg_latency=("latency_ms", "mean"), max_latency=("latency_ms", "max"))
        .sort_values("avg_latency", ascending=False)
        .reset_index()
    )
    fallback_2 = (
        events_df[events_df["derived_severity"].isin(["critical", "high"])]
        .groupby(["derived_severity", "category"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )
    fallback_3 = (
        events_df["endpoint_name"]
        .value_counts()
        .rename_axis("endpoint_name")
        .reset_index(name="count")
        .head(10)
    )
    fallback_sets = [fallback_1, fallback_2, fallback_3]
    for i, q in enumerate(spl_queries, start=1):
        search_summaries.append({
            "query_num": i,
            "sid": "fallback-local",
            "dispatch_state": "LOCAL_FALLBACK",
            "row_count": len(fallback_sets[i-1]),
            "elapsed_seconds": 0.0,
            "query": q,
            "error": "Splunk management API not reachable; displayed local analytical fallback."
        })
        search_results.append((f"Query {i}", fallback_sets[i-1]))

search_summary_df = pd.DataFrame(search_summaries)
display(search_summary_df)

for label, df in search_results:
    print(f"\n{label}")
    display(df.head(20))


## Architecture Comparison

The table below keeps the comparison tight and decision-oriented.


In [ ]:

comparison_df = pd.DataFrame([
    {
        "platform": "Splunk",
        "deployment_model": "Self-managed, Splunk Cloud, hybrid",
        "query_language": "SPL",
        "schema_enforcement": "Schema-on-read friendly for many search patterns",
        "ml_capabilities": "Mature ML Toolkit and anomaly workflows",
        "licensing_model": "Traditionally ingest-based; newer workload/flexible models also exist",
        "cost_at_100GB_day": "High relative cost; premium enterprise spend",
        "cloud_saas_option": "Yes",
        "citi_fit": "Strong for regulated financial services, auditability, RBAC, operational maturity",
    },
    {
        "platform": "ELK",
        "deployment_model": "Self-managed, Elastic Cloud, hybrid",
        "query_language": "Lucene / Query DSL / SQL-style options",
        "schema_enforcement": "Schema-on-write favored for performance and search quality",
        "ml_capabilities": "Good, especially in commercial Elastic features",
        "licensing_model": "Infrastructure + ops + optional commercial licensing",
        "cost_at_100GB_day": "Moderate if disciplined; can grow with storage/replicas/hot-warm tiers",
        "cloud_saas_option": "Yes",
        "citi_fit": "Good when customization and cost control matter more than packaged enterprise workflows",
    },
    {
        "platform": "Datadog",
        "deployment_model": "Primarily SaaS",
        "query_language": "Product-specific search/query UX across logs/APM/metrics",
        "schema_enforcement": "Structured ingestion with tagging discipline",
        "ml_capabilities": "Strong SaaS analytics, anomaly detection, Watchdog-style insights",
        "licensing_model": "Per host / per feature / per-GB logs",
        "cost_at_100GB_day": "Can become very high at scale, especially long retention + high-cardinality usage",
        "cloud_saas_option": "Yes",
        "citi_fit": "Excellent for cloud-native engineering teams needing fast correlation across signals",
    },
])

display(comparison_df)


## ELK deep dive

### What ELK really is

- **Elasticsearch** is the distributed search and indexing engine
- **Logstash** is the transformation and ingestion workhorse
- **Kibana** is the visualization and exploration layer
- **Beats** are lightweight edge shippers for logs, metrics, and events

### Why engineers like it

- Strong flexibility
- Infrastructure-funded instead of premium ingest licensing
- Deep control over cluster design, hot/warm/cold storage, retention, mappings, and shard strategy

### The tradeoff

ELK tends to reward teams that are good at platform engineering. You gain flexibility, but you also inherit:

- mapping decisions
- shard sizing
- index lifecycle management
- cluster tuning
- on-call responsibility

### Cost model

The clean story is **“infrastructure only”**, but the real cost is:

- compute
- storage
- replicas
- hot/warm/cold tiering
- engineering time
- operational support burden

### Important downside

For good performance, ELK often leans toward **schema-on-write** discipline. That is not inherently bad, but it means teams need stronger ingestion standards, naming consistency, and index management than they sometimes expect.


## Datadog deep dive

### What makes Datadog attractive

Datadog is strongest when the team wants one managed experience for:

- infrastructure monitoring
- APM
- distributed tracing
- logs
- dashboards
- alerting

### Why it wins fast

It is agent-based, fast to deploy, and very good at correlating:

- metrics
- traces
- logs

That correlation is the killer feature for modern cloud operations.

### Cost model

Datadog usually scales by combinations of:

- per-host pricing
- per-product pricing
- per-GB log pricing
- retention premiums
- high-cardinality tagging costs

### Strength

The fastest path to operational visibility for a cloud-native engineering org.

### Weakness

At scale, especially with high log volume and long retention, the spend can climb hard.


## Decision Framework

### Scenario

You have:

- **10TB/day** of telemetry logs
- a **20-person Data Engineering team**
- a **FinTech regulatory requirement for 7-year retention**

### Walk-through

#### If regulation and auditability are the center of gravity
Splunk is very hard to dismiss. It has a long enterprise track record, strong RBAC patterns, mature search workflows, and credibility in heavily governed operating models.

#### If cost is the primary hard boundary
ELK usually becomes the most rational answer. At 10TB/day, premium ingest economics can become painful. ELK lets the team design storage tiers, retention architecture, and cost controls more aggressively.

#### If cloud-native incident response speed matters most
Datadog is operationally elegant, but at **10TB/day plus 7-year retention**, cost usually becomes the first major objection.

### Practical answer for this scenario

- **Primary recommendation**: **ELK** for the log retention backbone, with disciplined architecture
- **Alternative**: **Splunk** for organizations that will explicitly pay for enterprise governance, search maturity, and reduced platform burden
- **Least likely fit at this scale**: **Datadog as the full long-retention log system of record**

### Citi-style interpretation

For a regulated financial institution, the most defensible answer is often:

- Splunk for premium regulated search and operations workflows **or**
- ELK for cost-controlled long-term retention with strong internal platform engineering

Datadog remains excellent, but more as a premium cloud operations layer than the cheapest long-horizon archive system at this data volume.


## What Just Happened

- **Splunk wins** in regulated financial services because of audit trail depth, RBAC maturity, search ergonomics, and long enterprise history.
- **ELK wins** when cost is the primary constraint and the team has the engineering maturity to run the platform well.
- **Datadog wins** for cloud-native APM and rapid correlation across metrics, traces, and logs.

That is the core mental model:

> **Splunk = premium enterprise operations**
>
> **ELK = flexible cost-controlled observability engineering**
>
> **Datadog = fast unified cloud-native visibility**


In [ ]:

show_section("Notebook summary artifacts")

summary = {
    "events_prepared": int(len(events_df)),
    "splunk_hec_reachable": bool(splunk_hec_up),
    "splunk_mgmt_reachable": bool(splunk_mgmt_up),
    "successful_hec_posts": int(ingest_result_df["ok"].fillna(False).sum()) if "ok" in ingest_result_df.columns else 0,
    "searches_run": int(len(search_summary_df)),
}
summary_df = pd.DataFrame([summary])
display(summary_df)
